# Kinsella Revisited — Pipeline Walkthrough

<a href="https://colab.research.google.com/github/nagehanrdogan/Kinsella-Revisited-A-Longitudinal-Replication/blob/main/notebooks/pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This notebook runs the full analysis pipeline step by step and displays
the resulting tables and figures inline, so you can inspect the data at
every stage instead of just running scripts blindly.

**Note on the raw data:** SIPRI's terms of use do not allow redistributing
the raw Trade Register export, so `data/raw/` is empty in this repository.
However, `data/processed/` already contains the cleaned data and
cross-section edge lists produced by Step 1, committed to the repo — so
**Steps 2-5 run out of the box**, with no extra download needed. Step 1
only runs if you've placed your own SIPRI export at `data/raw/1995-2025.csv`
(see the README for how to obtain it).

**Note on the figures:** the published Figures 1a-1e and 2 were drawn in R
(`src/06_figure1_network_maps.R`, `src/07_figure2_moving_window.R`), which
reproduce them exactly. Step 6 displays the committed PNGs; every number
behind them is computed in Python in Steps 2-5.

## Setup

Clones the repository when running on Google Colab (skipped if you're
already running this notebook locally inside a clone of the repo).

In [ ]:
import sys
from pathlib import Path

REPO_URL = "https://github.com/nagehanrdogan/Kinsella-Revisited-A-Longitudinal-Replication.git"
REPO_NAME = "Kinsella-Revisited-A-Longitudinal-Replication"

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    if not Path(REPO_NAME).exists():
        !git clone {REPO_URL}
    %cd {REPO_NAME}
    !pip install -q -r requirements.txt
    PROJECT_ROOT = Path.cwd()
else:
    # Running locally: assume this notebook lives in <project_root>/notebooks/
    PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

SRC_DIR = PROJECT_ROOT / "src"
# The numbered scripts import the shared formulas in src/kinsella_measures.py
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))
print(f"Project root: {PROJECT_ROOT}")
assert SRC_DIR.exists(), f"src/ not found at {SRC_DIR} — check PROJECT_ROOT"

In [ ]:
import importlib.util
import pandas as pd
from IPython.display import display, Image


def load_module(module_name: str, file_path: Path):
    """Loads one of the numbered src/*.py scripts as an importable module.
    (Their filenames start with a digit, so a plain `import` statement
    doesn't work.) Importing only defines functions -- it does not run
    main(), since __name__ inside the loaded module is not "__main__"."""
    spec = importlib.util.spec_from_file_location(module_name, file_path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module


clean_data = load_module("clean_data", SRC_DIR / "01_clean_data.py")
network_metrics = load_module("network_metrics", SRC_DIR / "02_network_metrics.py")
robustness_check = load_module("robustness_check", SRC_DIR / "03_robustness_check.py")
discussion_checks = load_module("discussion_checks", SRC_DIR / "04_discussion_checks.py")
moving_window = load_module("moving_window", SRC_DIR / "05_moving_window.py")

## Step 1 — Clean raw data (optional)

Only runs if you've placed a SIPRI export at `data/raw/1995-2025.csv`.
Otherwise this cell just reports that it's skipping, and the rest of the
notebook uses the already-processed data committed to the repo.

In [ ]:
if clean_data.RAW_PATH.exists():
    clean_data.main()
else:
    print(f"Skipping Step 1: no raw data found at {clean_data.RAW_PATH}")
    print("Using the already-processed data committed to data/processed/ instead.")

## Step 2 — Network metrics (Tables 1-4)

Density, centralization, out-degree centrality, HHI, and market share for
each of the 5 cross-sections (2005, 2010, 2015, 2020, 2025).

In [ ]:
edges = pd.read_csv(network_metrics.EDGES_PATH)

table1, table2 = network_metrics.compute_table1_and_2(edges)
table3, table4 = network_metrics.compute_table3_and_4(edges)

print("Table 1 — density & centralization")
display(table1)

In [ ]:
print("Table 2 — top 20 suppliers by out-degree centrality (2025 shown; change the year filter to inspect others)")
display(table2[table2["year"] == 2025])

In [ ]:
print("Table 3 — HHI & leading supplier's market share")
display(table3)

In [ ]:
print("Table 4 — top 10 suppliers by market share (2025 shown)")
display(table4[table4["year"] == 2025])

## Step 3 — Table 5 and scope checks

Table 5 recomputes density, centralization and HHI with component-level
ties (SIPRI's Engines and Sensors categories) removed. The two checks behind
it confirm the figures quoted in the text: the component share of total TIV
(6-10 per cent) and the share of ties involving unknown or non-state entries
(1.9-3.7 per cent).

In [ ]:
deliveries = pd.read_csv(robustness_check.DELIVERIES_PATH)
deliveries = robustness_check.classify(deliveries)

table5 = robustness_check.table5_platform_only(deliveries)
print("Table 5 — network measures with and without component-level ties")
display(table5)

In [ ]:
print("Component (engines + sensors) share of total TIV")
display(robustness_check.component_share_over_time(deliveries))

print("Ties and TIV involving unknown or non-state entries")
display(robustness_check.nonstate_share_over_time(deliveries))

## Step 4 — Discussion checks: Russia's in/out balance

How the balance between Russia's supplying and receiving roles narrows
across the five cross-sections, and who supplies Russia in 2025.

In [ ]:
balance = discussion_checks.russia_balance_over_time(edges)
print("Russia in/out balance over time")
display(balance)

In [ ]:
suppliers_2025 = discussion_checks.russia_2025_suppliers(edges)
print("Russia's 2025 suppliers")
display(suppliers_2025)

## Step 5 — Figure 2 data: five-year moving windows

Centralization and HHI recomputed on networks pooling all deliveries in
each five-year window, 2008-2025, labelled by the window's end year.

In [ ]:
figure2 = moving_window.moving_window_measures(pd.read_csv(moving_window.DELIVERIES_PATH))
print("Figure 2 data")
display(figure2.round(4))

## Step 6 — Figures 1a-1e and 2

Displays the published figures from `output/figures/`. To regenerate them,
run the R scripts after Steps 1-5 (needs tidyverse, igraph, ggraph and
ggrepel):

```
Rscript src/06_figure1_network_maps.R
Rscript src/07_figure2_moving_window.R
```

In [ ]:
FIGURES_DIR = PROJECT_ROOT / "output/figures"
for name in ["figure_1a_2005", "figure_1b_2010", "figure_1c_2015",
             "figure_1d_2020", "figure_1e_2025", "figure_2_moving_window"]:
    print(name)
    display(Image(filename=str(FIGURES_DIR / f"{name}.png"), width=600))

## Done

All tables now sit in `output/tables/` and all figures in
`output/figures/`, matching the numbers reported in the accepted version
of the article (see the README for the mapping from files to tables and
figures). If you spot a discrepancy against the paper, that's exactly what
this notebook is for — open an issue or a PR comment pointing at the
specific table/cell.